In [1]:
import os
import cv2
import numpy as np
import splitfolders
import random

# CHANGE THIS PATH
input_folder = r"C:\Users\Shishupal Kumar\OneDrive\Desktop\image\Datasets"
output_folder = r"C:\Users\Shishupal Kumar\OneDrive\Desktop\image\output_datasets"

# STEP 1: SPLIT DATASET INTO TRAIN / VAL / TEST
splitfolders.ratio(
    input_folder,
    output=output_folder,
    seed=42,
    ratio=(0.7, 0.2, 0.1)
)

print("✅ Dataset Split Completed!")

# STEP 2: AUGMENTATION FUNCTION
def augment_image(img):
    h, w = img.shape[:2]

    # Flip
    if random.random() > 0.5:
        img = cv2.flip(img, 1)

    # Rotate
    angle = random.randint(-20, 20)
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
    img = cv2.warpAffine(img, M, (w, h))

    # Brightness Safe Fix
    value = random.randint(-40, 40)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.int16)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] + value, 0, 255)
    hsv = hsv.astype(np.uint8)
    img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    # Blur
    if random.random() > 0.7:
        img = cv2.GaussianBlur(img, (5, 5), 0)

    return img



# STEP 3: AUGMENT ONLY TRAIN IMAGES
train_path = os.path.join(output_folder, "train")

for person_name in os.listdir(train_path):
    person_folder = os.path.join(train_path, person_name)

    for img_name in os.listdir(person_folder):
        img_path = os.path.join(person_folder, img_name)

        img = cv2.imread(img_path)
        if img is None:
            continue

        # Create 5 augmented images per original
        for i in range(5):
            aug_img = augment_image(img)
            new_name = img_name.split('.')[0] + f"_aug_{i}.jpg"
            save_path = os.path.join(person_folder, new_name)
            cv2.imwrite(save_path, aug_img)

print("✅ Augmentation Done! All names preserved.")


✅ Dataset Split Completed!
✅ Augmentation Done! All names preserved.
